# Amazon Bedrock Streaming Responses

This notebook demonstrates streaming responses from Amazon Bedrock, which is essential for creating responsive user experiences in real-time applications. Instead of waiting for the complete response, streaming allows processing text as it's generated.

## Benefits of Streaming:
- **Improved UX**: Users see responses immediately as they're generated
- **Lower Perceived Latency**: Text appears progressively rather than all at once
- **Real-time Processing**: Enable live applications like chatbots and assistants
- **Better Resource Utilization**: Process chunks as they arrive

## When to Use Streaming:
- Interactive chat applications
- Long-form content generation
- Real-time text processing
- Applications requiring immediate feedback

## Streaming API Concepts

The `invoke_model_with_response_stream` API returns an event stream instead of a complete response. Each event contains a chunk of the generated text.

### Key Components:
- **Event Stream**: Continuous flow of response chunks
- **Content Blocks**: Individual pieces of generated text
- **Delta Updates**: Incremental text additions
- **JSON Parsing**: Each chunk is a JSON object requiring parsing

### Stream Processing Pattern:
1. Make streaming API call
2. Iterate through event stream
3. Parse each JSON chunk
4. Extract text deltas
5. Process or display incrementally

In [1]:
import boto3
import json

# Initialize Bedrock client for streaming operations
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'amazon.nova-micro-v1:0'

# Prepare the request payload
# Same structure as regular invoke_model, but will return streaming response
payload = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": "Tell me what type of dances people do."
                }
            ]
        }
    ]
}

payload_json = json.dumps(payload)

# Use invoke_model_with_response_stream for streaming responses
response = bedrock.invoke_model_with_response_stream(
    modelId=model_id,
    body=payload_json,
    contentType='application/json',
    accept='application/json'
)

# Process the streaming response
stream = response['body']
full_text = ""

print("Receiving response stream:")
for event in stream:
    # Each event contains a chunk of the response
    chunk = event['chunk']
    chunk_str = chunk.get('bytes', b'').decode('utf-8')

    try:
        # Parse the JSON chunk to extract text content
        json_chunk = json.loads(chunk_str)

        # Look for content block deltas containing the actual text
        if "contentBlockDelta" in json_chunk:
            delta = json_chunk["contentBlockDelta"]["delta"]
            text = delta.get("text", "")
            full_text += text
            print(text, end='')  # Print immediately for real-time display

    except json.JSONDecodeError:
        # Skip malformed chunks (can happen with network issues)
        continue

Receiving response stream:
There are numerous types of dances performed around the world, each with its own unique style, cultural significance, and historical background. Here's a broad overview of some popular and notable dance forms:

### Classical and Traditional Dances
1. **Ballet**: A highly formalized dance style that originated in the Italian Renaissance courts and further developed in France and Russia. It's characterized by its precision and grace.
   
2. **Indian Classical Dances**:
   - **Bharatanatyam**: From Tamil Nadu, known for its elaborate gestures and facial expressions.
   - **Kathak**: Originating from North India, known for its intricate footwork and storytelling.
   - **Kathakali**: From Kerala, a dramatic dance-drama known for colorful makeup and expressive gestures.
   - **Kuchipudi**: From Karnataka, combines dance, drama, and abstract elements.

3. **Spanish Flamenco**: From Andalusia, characterized by passionate music, intricate footwork, and expressive danc

## Implementation Details

### Event Stream Structure:
The streaming response contains different types of events:
- **messageStart**: Indicates the beginning of the response
- **contentBlockStart**: Marks the start of a content block
- **contentBlockDelta**: Contains incremental text (what we process)
- **contentBlockStop**: Marks the end of a content block
- **messageStop**: Indicates the end of the complete response

### Error Handling:
- **JSON Parsing**: Some chunks may be malformed due to network issues
- **Connection Issues**: Implement retry logic for stream interruptions
- **Timeout Handling**: Set appropriate timeouts for long responses

### Performance Considerations:
- **Buffer Management**: Accumulate text for processing while displaying incrementally
- **Memory Usage**: For very long responses, consider processing chunks without storing all text
- **Network Efficiency**: Streaming reduces memory usage compared to waiting for complete responses

## Production Implementation Patterns

### Web Applications:
```python
# Example pattern for web streaming
def stream_to_client():
    for event in stream:
        # Process chunk
        text_chunk = extract_text(event)
        # Send to client via WebSocket or Server-Sent Events
        yield text_chunk
```

### Chat Applications:
- Use WebSockets for bidirectional communication
- Implement typing indicators during streaming
- Handle user interruptions gracefully

### Content Generation:
- Stream to temporary storage for long documents
- Implement progress indicators
- Allow users to stop generation early

### Monitoring and Observability:
- Track streaming latency metrics
- Monitor connection drop rates
- Log partial responses for debugging